## Ximea Post-Analysis — Filter, Undrift, Link

Takes the raw per-frame colour localisations from `Raw_Analysis.ipynb`
(`<FOV>.h5`, one row per detected spot per frame) through to a linked
single-molecule table, per FOV:

1. **Filter** — manual pandas threshold chain (style follows
   `notebooks/superres_dna_paint_cells/DNA_PAINT_Cells_PostAnalysis.ipynb`,
   values seeded from `Constants.FilteringConstants` since that notebook's own
   thresholds were tuned for a different, dense blinking DNA-PAINT sample).
2. **AIM undrift** — `DriftCorrectionFunctions.Drift_Correction_Functions().undrift(..., method='aim')`.
3. **Link with HDBSCAN** — `SM_extractionfunctions.extract_SMs().extract_single_molecules_HDBSCAN(...)`,
   used directly since the Ximea data has the full colour schema this method expects.

Beads are continuously visible across all 600 frames (not blinking), so
spatial HDBSCAN clustering of repeated same-position detections is the right
linking tool here — not the massive-cells notebook's temporal/dark-time
`link_localisations`, which is for intermittently-blinking emitters.

Preview cells tune parameters on one example FOV; the batch cell then runs
the same pipeline over every FOV.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from pathlib import Path

import sys
sys.path.append('../..')

from src import IOFunctions
from src.Constants import FilteringConstants
import DriftCorrectionFunctions as DCF
from src import SM_extractionfunctions

IO = IOFunctions.IO_Functions()
SM_E = SM_extractionfunctions.extract_SMs(camera='ximea')


In [ ]:
# ── Paths and acquisition parameters ────────────────────────────────────────────
XIMEA_FOLDER = Path('/scratch/sycamore_asap_server/ASAP_Members_Other_Imaging_Data/Brendan/20260624_Ximea_vs_Prime95_beads/100nm_beads_488nm_10mW_561_10perc_638_10mW_488LP_multinotch_Ximea')

PIXEL_SIZE_NM = 69.0
WIDTH, HEIGHT = 2064, 1544   # Ximea chip shape (confirmed via Camera_Calibrations/Ximea_Camera/gain.tif)

# AIM drift correction
AIM_SEGMENTATION = 20   # frames per drift segment (DNA_PAINT_Cells_PostAnalysis.ipynb convention)
AIM_INTERSECT_D  = 20 / PIXEL_SIZE_NM   # DriftConstants.AIM_INTERSECT_DISTANCE_NM / pixel_size
AIM_ROI_R        = 60 / PIXEL_SIZE_NM   # DriftConstants.AIM_ROI_RADIUS_NM / pixel_size

# HDBSCAN linking
MIN_CLUSTER_SIZE = 10

ximea_files = sorted(XIMEA_FOLDER.glob('*_MMStack_Pos-*.h5'))
# Only the raw per-frame fit results — exclude anything this notebook itself
# writes back into the same folder (see the save cell below).
ximea_files = [f for f in ximea_files if '_linked' not in f.name]
print(f'[Ximea] {len(ximea_files)} FOV .h5 files found')


### Preview — tune filter/undrift/link parameters on one FOV

In [ ]:
# ── Filtering thresholds — seeded from Constants.FilteringConstants ────────────
# Starting values for a bright bead sample; check the histograms below and adjust.
MAX_LOCALISATION_ERROR_PX = FilteringConstants.MAX_LOCALISATION_ERROR_PX   # 1.0 px
MAX_COLOUR_ERROR          = FilteringConstants.MAX_COLOUR_ERROR           # 0.15
MIN_SIGMA_PX              = FilteringConstants.MIN_SIGMA_NM / PIXEL_SIZE_NM
MAX_SIGMA_PX              = FilteringConstants.MAX_SIGMA_NM / PIXEL_SIZE_NM
MAX_SIGMA_ERROR_PX        = FilteringConstants.MAX_SIGMA_ERROR_NM / PIXEL_SIZE_NM
MIN_PHOTONS               = FilteringConstants.MIN_PHOTONS


def filter_ximea_locs(df):
    """Manual pandas threshold chain — style follows DNA_PAINT_Cells_PostAnalysis.ipynb,
    values seeded from FilteringConstants rather than that notebook's own
    (differently-tuned) numbers."""
    chi_val = np.median(df['chi_sqr'])
    df = df[df['chi_sqr'] < chi_val]
    df = df[(df['xc_err'] > 0) & (df['xc_err'] < MAX_LOCALISATION_ERROR_PX)]
    df = df[(df['yc_err'] > 0) & (df['yc_err'] < MAX_LOCALISATION_ERROR_PX)]
    df = df[(df['s_x'] > MIN_SIGMA_PX) & (df['s_x'] < MAX_SIGMA_PX)]
    df = df[(df['s_y'] > MIN_SIGMA_PX) & (df['s_y'] < MAX_SIGMA_PX)]
    df = df[df['s_x_err'] < MAX_SIGMA_ERROR_PX]
    df = df[df['s_y_err'] < MAX_SIGMA_ERROR_PX]
    for ch in ('A_B', 'A_G', 'A_R', 'bg_B', 'bg_G', 'bg_R'):
        df = df[df[f'{ch}_err'] < MAX_COLOUR_ERROR]
    df = df[df['photons'] > MIN_PHOTONS]
    return df.reset_index(drop=True)


example_fov = ximea_files[0]
raw_df = IO.read_h5_database(str(example_fov))
print(f'[{example_fov.name}] {len(raw_df):,} raw localisations')

filt_df = filter_ximea_locs(raw_df)
print(f'[{example_fov.name}] {len(filt_df):,} after filtering ({len(filt_df)/len(raw_df):.1%})')

fig, axs = plt.subplots(1, 3, figsize=(11, 3))
axs[0].hist(raw_df['chi_sqr'], 100, alpha=0.5, label='raw')
axs[0].hist(filt_df['chi_sqr'], 100, alpha=0.5, label='filtered')
axs[0].set_xlabel('chi_sqr'); axs[0].legend()
axs[1].hist(raw_df['photons'], 100, range=(0, 20000), alpha=0.5, label='raw')
axs[1].hist(filt_df['photons'], 100, range=(0, 20000), alpha=0.5, label='filtered')
axs[1].set_xlabel('photons')
axs[2].hist(raw_df['A_R'], 100, alpha=0.5, color='red', label='A_R')
axs[2].hist(raw_df['A_G'], 100, alpha=0.5, color='green', label='A_G')
axs[2].hist(raw_df['A_B'], 100, alpha=0.5, color='blue', label='A_B')
axs[2].set_xlabel('colour fraction'); axs[2].legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── AIM undrift preview ─────────────────────────────────────────────────────────
info = [{
    'Width': WIDTH, 'Height': HEIGHT,
    'Frames': int(filt_df['frame'].max()),
    'Pixelsize': PIXEL_SIZE_NM,
}]

drift_corrector = DCF.Drift_Correction_Functions()
corrected_locs, drift_result = drift_corrector.undrift(
    locs=filt_df.to_records(index=False),
    info=info,
    method='aim',
    segmentation=AIM_SEGMENTATION,
    intersect_d=AIM_INTERSECT_D,
    roi_r=AIM_ROI_R,
)
corrected_df = pd.DataFrame(corrected_locs)

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(drift_result.drift_x * PIXEL_SIZE_NM, label='drift x')
ax.plot(drift_result.drift_y * PIXEL_SIZE_NM, label='drift y')
ax.set_xlabel('segment'); ax.set_ylabel('drift / nm'); ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ── HDBSCAN linking preview ─────────────────────────────────────────────────────
single_molecule_db, single_frame_db = SM_E.extract_single_molecules_HDBSCAN(
    corrected_df, min_cluster_size=MIN_CLUSTER_SIZE,
)
print(f'[{example_fov.name}] {len(single_molecule_db)} linked single molecules from '
      f'{len(single_frame_db)} assigned localisations (of {len(corrected_df)} undrifted)')

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(single_frame_db['xc'], single_frame_db['yc'], s=1, alpha=0.3,
           c=single_frame_db['molecular_index'], cmap='tab20')
ax.scatter(single_molecule_db['xc'], single_molecule_db['yc'], s=20, marker='+', color='black')
ax.set_aspect('equal')
ax.set_title(f'{example_fov.name}\n{len(single_molecule_db)} molecules')
plt.tight_layout()
plt.show()


### Full batch — run filter/undrift/link over every FOV

In [ ]:
# ── Full fit — every Ximea FOV ──────────────────────────────────────────────────
for i_fov, fov_path in enumerate(ximea_files):
    sm_path     = fov_path.with_name(fov_path.stem + '_linked_sm.h5')
    frames_path = fov_path.with_name(fov_path.stem + '_linked_frames.h5')
    if sm_path.exists() and frames_path.exists():
        print(f'[{i_fov + 1:3d}/{len(ximea_files)}] {fov_path.name}  (skipped, already linked)')
        continue

    raw_df  = IO.read_h5_database(str(fov_path))
    filt_df = filter_ximea_locs(raw_df)
    if len(filt_df) < MIN_CLUSTER_SIZE:
        print(f'[{i_fov + 1:3d}/{len(ximea_files)}] {fov_path.name}  '
              f'only {len(filt_df)} locs after filtering, skipping')
        continue

    info = [{
        'Width': WIDTH, 'Height': HEIGHT,
        'Frames': int(filt_df['frame'].max()),
        'Pixelsize': PIXEL_SIZE_NM,
    }]
    drift_corrector = DCF.Drift_Correction_Functions()
    corrected_locs, drift_result = drift_corrector.undrift(
        locs=filt_df.to_records(index=False), info=info, method='aim',
        segmentation=AIM_SEGMENTATION, intersect_d=AIM_INTERSECT_D, roi_r=AIM_ROI_R,
    )
    corrected_df = pd.DataFrame(corrected_locs)

    single_molecule_db, single_frame_db = SM_E.extract_single_molecules_HDBSCAN(
        corrected_df, min_cluster_size=MIN_CLUSTER_SIZE,
    )

    IO.write_h5_database(single_molecule_db, str(sm_path), normalise_photons=False)
    IO.write_h5_database(single_frame_db, str(frames_path), normalise_photons=False)

    print(f'[{i_fov + 1:3d}/{len(ximea_files)}] {fov_path.name}  '
          f'{len(raw_df):,} -> {len(filt_df):,} filtered -> '
          f'{len(single_molecule_db)} linked molecules')

print('[Ximea] Post-analysis done.')
